# Agentic Pitch Coaching System
 *Author: Nguyen Bui*

 *Date: 2026-08-10*


## Business Context

As a junior ML engineer (MLE) at Pitchwise, a small startup accelerator where you have been asked to prototype an agentic pitch coaching system the partners can use to triage and improve founder pitches in the weeks before demo day.

#### 1. Company and Context

Pitchwise is a small startup accelerator based in Pittsburgh, founded in 2019 by two former operators. One of them built and sold a consumer hardware company, the other had a first startup that ran out of money before it found a market. They like to say that between them they have been to the moon and they have been to the floor, and that is exactly why founders trust them.

The accelerator runs two cohorts a year of about fifteen companies each. Pitchwise's whole pitch (no pun intended) is that the founders running it have actually sat on both sides of the table. They are opinionated about what makes a pitch land and what makes a pitch flop, and they have written those opinions down in a playbook that nobody outside the firm has ever seen.

The eight weeks leading up to demo day are the most intense part of the program. Every founder in the cohort goes through dozens of practice pitch sessions in front of the partners, who do not pull punches. One of the partners is known for closing her laptop ten seconds into a bad opening line and saying "try again." The other has a phrase he uses so often the cohort prints it on T-shirts: "I do not understand what your company does and I have been listening for two minutes." Founders dread these sessions and they love them, because the bar at Pitchwise is high and the partners' instincts are real.

#### 2. Business Challenge

This year's cohort is bigger than usual, and the partners are drowning. Drafts are piling up faster than the partners can review them, and feedback quality is starting to slip because the partners are rushing through review sessions. The team has been asked whether an AI-assisted system could handle the first pass on pitch drafts, so the partners can focus their time on the second-pass review where their judgment actually adds value.

The partners are skeptical of generic AI coaching tools. They have tried two off-the-shelf products and were unimpressed: the feedback was generic and did not reflect what they actually care about. They want a system that is grounded in the Pitchwise playbook (their own rubric, their own standards, their own opinions) and not in whatever the model picked up from the open internet.

#### 3. Business Goal

The goal is to build a prototype agentic system that can take a rough draft of a founder's pitch and return a polished version. The system should:

- Recognize what kind of company is pitching. The partners have three loose categories: consumer products (sold directly to individual people), B2B SaaS (software sold to other businesses on a subscription), and deep tech (companies built around a specific piece of novel technology or science). Each category gets evaluated against a different rubric.
- Iterate on the pitch, scoring it against the matched rubric and revising it until it either meets the bar or has stopped improving.
- Ground its evaluation in the actual Pitchwise playbook, not in the model's own opinions about pitches.

If the prototype works, the partners will use it as a first-pass screen so they can spend their review time on the pitches that have already reached a baseline quality.

#### 4. Role and Task

The partners have given you their playbook (the three rubrics, one per pitch type) and three sample founder pitches that came in this week. Your job is to build the prototype and run it end-to-end on those three pitches so the partners can see what the system actually does.

This work requires judgment at every stage. The classifier prompt determines whether the right rubric gets pulled. The evaluator's instructions determine whether the model takes the playbook seriously or talks around it. The iteration cap determines whether you waste API calls on pitches that are not going to converge. The decisions you make about how to combine these pieces will determine whether the partners trust the system enough to use it.


## Import

In [1]:
import openai_setup
from pydantic import BaseModel
from typing import Literal
from agents import Agent, Runner, set_tracing_disabled
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

set_tracing_disabled = True

import sys
if sys.platform=="win32":
    import os
    import mcp.client.stdio
    mcp.client.stdio.stdio_client.__wrapped__.__defaults__ = (open(os.devnull, "w"),)

## Connect to Pitchwise MPC server + List tools


In [2]:
async def inspect_server():
    server = MCPServerStdio(
        params = {"command": "python", "args": ["pitchwise_server.py"]},
        client_session_timeout_seconds=60
    )

    async with server:
        tools = await server.list_tools()
        print("="*60)
        print("PITCHWISE MCP SERVER - Tool Inspection")
        print("="*60)
        print(f"\nDiscovered {len(tools)} tool(s):\n")

        for tool in tools:
            print(f"TOOL: {tool.name}")
            print(f"Description: {tool.description}")
            if hasattr(tool, "inputSchema") and tool.inputSchema:
                schema = tool.inputSchema
                if "properties" in schema:
                    print("Parameters:")
                    for pname, pinfo in schema["properties"].items():
                        ptype = pinfo.get("type", "unknown")
                        required = pname in schema.get("required", [])
                        marker = " (required)" if required else " (optional)"
                        print(f" - {pname}: {ptype}{marker}")
            print()
await inspect_server()


PITCHWISE MCP SERVER - Tool Inspection

Discovered 1 tool(s):

TOOL: get_rubric
Description: 
    Return the Pitchwise scoring rubric for a given pitch type.

    Args:
        pitch_type: One of 'consumer', 'b2b_saas', or 'deep_tech'.

    Returns:
        dict: A dictionary with 'name' and 'criteria' keys, or
              {'name': 'Unknown', 'criteria': []} if pitch_type is not recognized.
    



## Classifier Agent

In [5]:
class PitchClassification(BaseModel):
    pitch_type: Literal["consumer", "b2b_saas", "deep_tech"]    #
    reasoning: str

classifier_agent = Agent(
    name = "Pitch Classifier",
    instructions =
    """
    You are classifying a startup pitch into one of 3 categories:
    - consumer: A product sold directly to indivisual people (physical goods, subscription boxes, apps for personal use).
    - b2b_saas: Software sold to other businesses on a subscription basis.
    - deep_tech: A company whose core advantage comes from a specific piece of novel technology or science (hardware sensors, materials science, robotics, specialized AI infrastructure).
    Read the draft pitch and decie which category is the best fit.
    """,
    model = "gpt-4o-mini",
    output_type = PitchClassification,
)

PITCH_TYPE_LABELS = {
    "consumer": "Consumer Product",
    "b2b_saas": "B2B SaaS",
    "deep_tech": "Deep Tech",
}

### Test on examples

*(edited)* Tests passed

In [7]:
EXAMPLE_PITCH_CONSUMER = """We are MugMail, a monthly subscription that ships a different small-batch coffee mug to your door each month, paired with the coffee that inspired it."""
        
EXAMPLE_PITCH_B2B = """We are LedgerLite, a billing tool for solo accountants.They subscribe and use it to send invoices and track unpaid bills for their small business clients."""

for label, pitch in [("Example A", EXAMPLE_PITCH_CONSUMER),("Example B", EXAMPLE_PITCH_B2B)]:
    result = await Runner.run(classifier_agent, pitch)
    classification = result.final_output
    print(f"------{label}------")
    print(f"Pitch: {pitch}")
    print(f"Classified as: {classification.pitch_type} ({PITCH_TYPE_LABELS[classification.pitch_type]})")
    print(f"Reasoning: {classification.reasoning}")
    print()


------Example A------
Pitch: We are MugMail, a monthly subscription that ships a different small-batch coffee mug to your door each month, paired with the coffee that inspired it.
Classified as: consumer (Consumer Product)
Reasoning: MugMail offers a subscription service targeting individual consumers who enjoy coffee and unique mugs, which fits the consumer category.

------Example B------
Pitch: We are LedgerLite, a billing tool for solo accountants.They subscribe and use it to send invoices and track unpaid bills for their small business clients.
Classified as: b2b_saas (B2B SaaS)
Reasoning: The pitch describes a billing tool specifically for solo accountants, indicating it is software designed for business use on a subscription basis.



## Coach Agent

In [25]:
consumer_coach = Agent(
    name = "Consumer Coach",
    instructions = 
    """You are a pitch coach for Pitchwise specializing in consumer products.
    When given a founder's draft pitch, rewrite it so that it does 3 things well:
        1. Opens with a specific customer moment (not a category description or a market statistic).
        2. Explains why now: what changed in the world that makes this product possible or necessary today.
        3. Names a concrete distribution channel and explains why the product will travel through it.

    IMPORTANT: DO NOT INVENT FACTS that are NOT in the founder's draft.
        If the draft does not contain the information a criterion needs (for example, no distribution channel is mentioned), 
        write a clear placeholder in brackets (for example [DISTRIBUTION CHANNEL NEEDED: founder must name a concrete channel]), instead of fill in or make one up.
        It is the founder's job to fill those in, not yours.

    Return only the rewritten pitch. No reample, no bullet list, no commentary.
    """,
    model = "gpt-4o-mini",
)

b2b_saas_coach = Agent(
    name = "B2B SaaS Coach",
    instructions = 
    """You are a pitch coach for Pitchwise specializing in B2B SaaS.
    When given a founder's draft pitch, rewrite it so that it does 3 things well:
        1. Describes a real, named workflow the buyer dislike today (not a vague "inefficiency").
        2. Gives a concrete number of potential buyers and explains how that number was estimated.
        3. Cites a specific traction signal (a pilot, a paying customer, a letter of intent) instead of generic "strong interest".
    
    IMPORTANT: DO NOT INVENT FACTS that are NOT in the founder's draft.
        If the draft does not contain the information a criterion needs (for example, no traction signal is mentioned), 
        write a clear placeholder in brackets (for example [TRACTION SIGNAL NEEDED: founder must cite a specific pilot, paying customer, or letter of intent]), instead of fill in or make one up.
        It is the founder's job to fill those in, not yours.
    
    Return only the rewritten pitch. No reample, no bullet list, no commentary.
    """,
    model = "gpt-4o-mini",
)

deep_tech_coach = Agent(
    name = "Deep Tech Coach",
    instructions = 
    """You are a pitch coach for Pitchwise specializing in deep tech.
    When given a founder's draft pitch, rewrite it so that it does 3 things well:
        1. Describes the team technical edge (what they can do that no one else can) in language a non-specialist can understand, no jargons.
        2. Names what protects the company from being copied (patents, proprietary data, specialized talent, hard-to-replicate process) and how long that protection holds.
        3. Identifies a specific first customer who would pay for an early version of the product, not a vague "future market".
    
    IMPORTANT: DO NOT INVENT FACTS that are NOT in the founder's draft.
        If the draft does not contain the information a criterion needs (for example, no first customer is mentioned), 
        write a clear placeholder in brackets (for example [FIRST CUSTOMER NEEDED: founder must name a specific first buyer]), instead of fill in or make one up.
        It is the founder's job to fill those in, not yours.
    
    Return only the rewritten pitch. No reample, no bullet list, no commentary.
    """,
    model = "gpt-4o-mini",
)


### Test on example

*(edited)* Instructions are clear and final draft sounds good

In [10]:
EXAMPLE_PITCH_DEEP_TECH = "We are FieldOptic, a startup using novel optical sensors to monitor crop health. Our technology is better than existing sensors. We believe the agriculture market is huge and we are well positioned to win."

result = await Runner.run(deep_tech_coach, EXAMPLE_PITCH_DEEP_TECH)
print("------ Original draft ------")
print(EXAMPLE_PITCH_DEEP_TECH)
print()
print("------ Rewritten by Coach ------")
print(result.final_output)

------ Original draft ------
We are FieldOptic, a startup using novel optical sensors to monitor crop health. Our technology is better than existing sensors. We believe the agriculture market is huge and we are well positioned to win.

------ Rewritten by Coach ------
At FieldOptic, we leverage advanced optical sensors that provide unique insights into crop health—insights that traditional sensors simply can't deliver. Our team's deep expertise in sensor technology allows us to create highly sensitive, real-time monitoring solutions that enable farmers to make informed decisions to optimize their yields.

We have secured several patents on our sensor designs and the algorithms that process the data, protecting our innovations for the next 20 years. This means our technology isn't just advanced; it is safeguarded against competitors looking to replicate our success.

We're actively targeting [FIRST CUSTOMER NEEDED: founder must name a specific first buyer], who is eager to utilize our e

# Evaluator-Optimizer Workflow
*Built with OpenAI Agents SDK and LangGraph*

The evaluator-optimizer loop should have 3 agents:
- **Generator**: the coach produces the initial draft
- **Evaluator**: get MCP tool `get_rubric`, list per-criterion feedback string
- **Optimizer**: takes the current draft and the evaluator's per-criterion feedback, and rewrites the pitch to address every flagged weakness

## Pitch Evaluation Schema

In [28]:
class PitchEvaluation(BaseModel):
    passes: bool
    per_criterion_feedback: list[str]
    summary:str

EVALUATOR_INSTRUCTIONS = """You are a Pitchwise evaluator. You are given a pitch_type and a pitch draft.

1. Call the get_rubric tool EXACTLY once with the given pitch_type, after receiving its result, immediately produce your final output; do not call it or any other tool again.
    Retrieve the 3 criteria for this kind of pitch.
    DO NOT rely on your own judgment about what a good pitch looks like. The rubric returned by the tool is the ONLY standard.
2. Score the draft against each of the three criteria from the rubric, in the order that they were returned.
3. Set passes = True only if all 3 criteria are clearly met. If even 1 criterion is not clearly met, set passes = False.
4. Return EXACTLY 3 entries in per_criterion_feedback, 1 per rubric criterion, in the same order as the rubric.
    Each entry should name the criterion it addresses and state specifically what the pitch does well or poorly on that criterion.
5. Write a one-sentence summaryof the overall verdict.

Ground every judgement in the rubric criteria returned by the tool. DO NOT introduce criteria that are not in the rubric, and do not skip calling the tool.
"""

In [13]:
optimizer_agent = Agent(
    name = "Pitch Optimizer",
    instructions = """You are a pitch revision specialist for Pitchwise. You will receive the most recent draft of a pitch and the evaluator's per-criterion feedback.
    Rewrite the pitch so that it directly addresses every piece of feedback that flagged a weakness. Keep the founder's voice and any details that already worked.
    
    IMPORTANT: DO NOT invent facts that are not in the current draft. If the evaluator flagged a missing piece of information (for example, no first customer is named), do not make one up.
        Instead, write a clear bracketed placeholder like [FIRST CUSTOMER NEEDED: founder must name a specific buyer] so the founder knows what to fill in. Sharpening vague language is good. Inventing customers, numbers, partnerships, or technical claims is not.
    
    Return only the rewritten pitch. No preamble, no bullet list, no commentary.
    """,
    model = "gpt-4o-mini"
)

## Test on example
The evaluator should read a rubric from the MCP server, and that it produces the structured feedback defined. Output: `passes = True`, 3 per-criterion feedback, 1 sentence summary.

In [14]:
EXAMPLE_EVAL_PITCH = (
    "On a Saturday morning at the farmers market, a new dog owner picks up a SniffBox sample "
    "and her puppy chooses the air-dried sweet potato chew over every other treat on the table. "
    "Pet food labeling rules tightened in California last year, and owners now want treats with "
    "ingredient lists they can actually read. SniffBox partners with independent veterinary "
    "clinics, who hand the boxes to first-time puppy owners during their initial visit, turning "
    "a worried vet appointment into a memorable first treat."
)

async def test_evaluator():
    server = MCPServerStdio(
        params = {
            "command": "python",
            "args": ["pitchwise_server.py"]
        },
        client_session_timeout_seconds = 60,
    )

    async with server:
        evaluator_agent = Agent(
            name = "Pitch Evaluator",
            instructions = EVALUATOR_INSTRUCTIONS,
            model = "gpt-4o-mini",
            output_type = PitchEvaluation,
            mcp_servers=[server],
        )

        eval_input = f"Pitch type: consumer\n\nPitch:\n{EXAMPLE_EVAL_PITCH}"
        result = await Runner.run(evaluator_agent, eval_input)
        evaluation = result.final_output
        print("------ Evaluator output (structured) ------")
        print(f"passes: {evaluation.passes}")
        print(f"summary: {evaluation.summary}")
        print("per_criterion_feedback:")
        for i, item in enumerate(evaluation.per_criterion_feedback, 1):
            print(f"  {i}. {item}")

await test_evaluator()


------ Evaluator output (structured) ------
passes: True
summary: The pitch successfully meets all evaluation criteria.
per_criterion_feedback:
  1. Hook: The pitch effectively opens with a vivid and relatable customer moment, showcasing a new dog owner’s experience at the farmers market, which engages the audience right from the start.
  2. Why now: The pitch clearly explains that recent changes in pet food labeling rules in California have made it necessary for owners to seek out more transparent and understandable ingredients, creating urgency for the product.
  3. Distribution: The pitch names independent veterinary clinics as a concrete distribution channel, explaining how they turn vet visits into positive experiences for first-time puppy owners by providing the SniffBox.


## Evaluator-Optimizer Pipeline

1. The classifier picks the pitch type.
2. Inside an `async with MCPServerStdio(...) as server:` block, the evaluator agent is constructed with `mcp_servers=[server]` so it can call the `get_rubric` tool.
3. The matched coach drafts version 1.
4. The evaluator scores it.
5. If the evaluator passes the pitch, the loop ends. Otherwise the optimizer revises and the evaluator scores the new version.
6. The loop runs until the evaluator returns `passes=True` or it hits a hard iteration cap of 3.

In [23]:
MAX_ITERATIONS = 3

async def pitch_pipeline(pitch_id: str, pitch_text: str) -> dict:
    """
    Run a draft pitch through the full Pitchwise workflow:
    classify -> coach -> evaluator-optimizer loop -> return final pitch
    """

    # =========== CLASSIFY THE PITCH ===========
    classification_result = await Runner.run(classifier_agent,pitch_text)
    classification = classification_result.final_output
    pitch_type = classification.pitch_type
    print(f"\n====== {pitch_id}: classified as {PITCH_TYPE_LABELS[pitch_type]} ======")
    print(f"Reasoning: {classification.reasoning}\n")


    # ========= CONNECT TO MCP SERVER ===========
    server = MCPServerStdio(
        params = {
            "command": "python",
            "args": ["pitchwise_server.py"]
        },
        client_session_timeout_seconds = 60,
    )

    async with server:
        evaluator_agent = Agent(
            name = "Pitch Evaluator",
            instructions = EVALUATOR_INSTRUCTIONS,
            model = "gpt-4o-mini",
            output_type = PitchEvaluation,
            mcp_servers=[server],
        )

        if pitch_type == "consumer":
            coach = consumer_coach
        elif pitch_type == "b2b_saas":
            coach = b2b_saas_coach
        else:
            coach = deep_tech_coach

        coach_result = await Runner.run(coach, pitch_text)
        current_draft = coach_result.final_output
        evalution = None

        for iteration in range(1, MAX_ITERATIONS+1):
            eval_input = f"Pitch type: {pitch_type}\n\nPitch:\n{current_draft}"
            eval_result = await Runner.run(evaluator_agent, eval_input)
            evaluation = eval_result.final_output
            print(f"--- Iteration {iteration} ---")
            print(f"Summary: {evaluation.summary}")
            if evaluation.passes:
                break
            if iteration < MAX_ITERATIONS:
                feedback_text = "\n".join(evaluation.per_criterion_feedback)
                optimizer_input = (
                    f"Current draft:\n{current_draft}\n\n"
                    f"Evaluator feedback:\n{feedback_text}\n\n"
                    "Revise the draft to address this feedback."
                )
                optimizer_result = await Runner.run(optimizer_agent, optimizer_input)
                current_draft = optimizer_result.final_output
            pass
    return {
        "pitch_type": pitch_type,
        "final_draft": current_draft,
        "evaluation": evaluation,
        "iterations": iteration,
    }


## Test Workflow

In [29]:
PITCH_A = """We are SnackPocket, a monthly snack subscription box for kids ages 6 to 12. 
Last Tuesday a mom in our pilot group texted us a photo of her seven-year-old reading the SnackPocket 
ingredient label out loud at the breakfast table, sounding out each word. That moment is why parents 
keep telling us they want better snacks for their kids but get exhausted reading every label themselves 
at the grocery store. Three big school districts in our region tightened their snack guidelines last year, 
so the bar for what kids are allowed to bring to lunch is suddenly higher and parents are scrambling. 
We curate boxes built around real-food ingredients and ship monthly with a kid-friendly trading card 
inside each box. We are launching through three school PTA fundraisers next month, which already reach 
hundreds of families per school and have a built-in referral hook for parents."""

PITCH_B = """ShiftRoster is software that helps small businesses manage employee schedules.
Owners of small restaurants and retail shops spend three to five hours every Sunday night rebuilding 
next week's shift schedule in spreadsheets, juggling time-off requests, double-booked employees, and 
last-minute swaps. Our product lets them generate a full week's schedule in under five minutes by 
dragging and dropping employees onto a calendar. There are roughly 700,000 small restaurants and 
retail businesses in the US with five to fifty hourly employees, which is the band where spreadsheet 
scheduling breaks down but full enterprise tools are overkill. We have strong interest from local 
restaurants and retail shops in our city and plan to launch in the next quarter."""

PITCH_C = """KitchenSense is a next-generation IoT platform leveraging AI to revolutionize restaurant
 operations. Using our proprietary sensor technology and machine learning algorithms, we deliver 
 actionable insights that drive operational excellence across the food service vertical. With the 
 rapid digital transformation of the restaurant industry, our addressable market is massive. We are 
 uniquely positioned to capture significant share."""

PITCH_D = """We are FreshFold, a monthly clothing-repair kit for busy parents. On Sunday night, 
a parent discovers three school shirts with missing buttons and no time to visit a tailor before Monday morning.
More families are buying fewer, better-quality clothes and want to extend their lifespan instead of replacing them. 
FreshFold delivers pre-measured repair materials, simple video instructions, and seasonal kits through 
direct-to-consumer subscriptions promoted by parenting newsletters and family-oriented creators."""

PITCH_E = """InvoicePilot helps independent construction companies collect overdue invoices. Office managers 
currently export project data from accounting software, search email for payment promises, and call customers 
one by one every Friday afternoon. There are approximately 185,000 US construction firms with 10 to 100 employees, 
based on Census business counts, and we estimate 60,000 have enough recurring invoices to need dedicated 
collections software. Two regional contractors are paying for a six-week pilot, and both have used InvoicePilot 
to automate reminder sequences and track promised payment dates."""

PITCH_F = """We are AeroWeave, developing lightweight structural panels made from a proprietary fiber-laying 
process for electric delivery vans. Our technical edge is a manufacturing method that places reinforcing fibers 
only where each panel needs strength, reducing weight without requiring thicker material. The process is 
protected by a pending patent application and specialized production know-how that we expect to take competitors 
several years to reproduce. FleetWorks, a regional electric-van manufacturer, is evaluating an early panel for 
its next prototype vehicle and has agreed to pay for a materials test."""

In [30]:
results = {}

for pitch_id, pitch_text in [("PITCH A", PITCH_A), ("PITCH B", PITCH_B), ("PITCH C", PITCH_C), ("PITCH D", PITCH_D), ("PITCH E", PITCH_E), ("PITCH F", PITCH_F)]:
    result = await pitch_pipeline(pitch_id, pitch_text)
    results[pitch_id] = result
    print(f"\n--- {pitch_id} FINAL ({result['iterations']} iteration(s), passed={result['evaluation'].passes}) ---")
    display(Markdown(f"**Final draft:**\n\n{result['final_draft']}"))
    print(f"\nFinal verdict: {result['evaluation'].summary}")
    print(f"\nPer-criterion feedback from the last evaluation:")
    for i, item in enumerate(result['evaluation'].per_criterion_feedback, 1):
        print(f"  {i}. {item}")
    print("=" * 70)



====== PITCH A: classified as Consumer Product ======
Reasoning: The startup offers a monthly snack subscription box specifically targeting kids, selling directly to individual consumers (parents) who are looking for healthy snack options for their children.

--- Iteration 1 ---
Summary: The pitch effectively meets all three criteria, showcasing a strong emotional connection, urgency in the market, and a clear distribution strategy.

--- PITCH A FINAL (1 iteration(s), passed=True) ---


**Final draft:**

Last Tuesday, a mom in our pilot group shared a heartfelt moment with us—a photo of her seven-year-old reading the SnackPocket ingredient label out loud at the breakfast table, proudly sounding out each word. This is the moment that drives parents to seek out better snacks for their kids, yet they often feel overwhelmed when trying to decipher every label at the grocery store. With three major school districts in our region tightening their snack guidelines last year, the standards for what kids can bring to lunch have suddenly increased, leaving parents scrambling for options. We curate monthly boxes filled with real-food ingredients, each accompanied by a fun trading card that engages kids. We're launching through three school PTA fundraisers next month, which already connect us to hundreds of families per school and leverage a built-in referral system for parents.


Final verdict: The pitch effectively meets all three criteria, showcasing a strong emotional connection, urgency in the market, and a clear distribution strategy.

Per-criterion feedback from the last evaluation:
  1. Hook: The pitch successfully opens with a specific customer moment by sharing a relatable story of a mom and her child interacting with the product, which captures the emotional aspect of choosing snacks.
  2. Why now: It effectively explains the recent changes in school snack guidelines, highlighting the urgency for parents to find better snack options for their kids at this time.
  3. Distribution: The pitch names PTA fundraisers as a concrete distribution channel, detailing how it connects to families and enables word-of-mouth referrals.

====== PITCH B: classified as B2B SaaS ======
Reasoning: The pitch is focused on providing scheduling software to small businesses, specifically aimed at restaurant and retail owners. It addresses their pain points related to employe

**Final draft:**

ShiftRoster is software that helps small businesses manage employee schedules. Owners of small restaurants and retail shops, like [SPECIFIC RESTAURANT NAME] and [SPECIFIC RETAIL SHOP NAME], often spend three to five hours every Sunday night rebuilding next week's shift schedule in spreadsheets, grappling with challenges such as time-off requests, double-booked employees, and last-minute swaps. For example, [DETAILED SCENARIO THAT ILLUSTRATES A FRUSTRATING WORKFLOW]. Our product simplifies this chaotic process by allowing users to generate a full week's schedule in under five minutes with a drag-and-drop calendar interface. We estimate there are approximately 700,000 small restaurants and retail businesses in the US with five to fifty hourly employees, derived from industry reports and government data on small business demographics. We have secured a pilot agreement with [FIRST CUSTOMER NEEDED: founder must name a specific buyer], and we plan to launch in the next quarter.


Final verdict: Overall, the pitch does not clearly meet all the criteria, particularly in naming a specific scenario and buyer.

Per-criterion feedback from the last evaluation:
  1. Customer pain: The pitch provides a real workflow that small business owners dislike and illustrates the frustrations with scheduling, but lacks a specific named scenario that showcases this issue.
  2. Market sizing: The pitch estimates the number of potential buyers based on industry reports and government data, which is good, but does not provide an exact number or detailed methodology for the estimation.
  3. Traction: The pitch mentions a pilot agreement but fails to name a specific buyer, hence lacking a specific signal of traction.

====== PITCH C: classified as B2B SaaS ======
Reasoning: KitchenSense provides an IoT platform designed to optimize restaurant operations, which indicates it is targeting business clients (restaurants) rather than individual consumers. The emphasis on proprietary techno

...


Final verdict: The pitch successfully meets all three criteria of the rubric.

Per-criterion feedback from the last evaluation:
  1. Hook: The pitch effectively opens with a relatable scenario depicting a common challenge faced by parents, which engages the audience's attention immediately.
  2. Why now: It clearly explains the current trend towards sustainable living and the desire among parents to repair rather than replace clothing, making the product timely and relevant.
  3. Distribution: The pitch specifies a direct-to-consumer subscription model via their website, highlighting a convenient channel that aligns with current shopping behaviors of busy families.

====== PITCH E: classified as B2B SaaS ======
Reasoning: InvoicePilot is a software solution designed to assist independent construction companies in automating the collections process for overdue invoices. The target customers are businesses, specifically construction firms, which aligns with the B2B SaaS model. This soft

**Final draft:**

InvoicePilot helps independent construction companies collect overdue invoices. Office managers currently export project data from accounting software, search email for payment promises, and make multiple phone calls to customers every Friday afternoon—an exhausting workflow that leads to frustration and delayed payments. There are approximately 185,000 US construction firms with 10 to 100 employees, based on Census business counts, and we estimate that around 60,000 of these companies have enough recurring invoices to warrant dedicated collections software. Two regional contractors are currently paying for a six-week pilot, and both have successfully used InvoicePilot to automate reminder sequences and track promised payment dates.


Final verdict: The pitch meets all the rubric criteria successfully.

Per-criterion feedback from the last evaluation:
  1. Customer pain: The pitch effectively describes the frustrating workflow office managers face with overdue invoices, clearly identifying their pain points.
  2. Market sizing: The pitch provides concrete numbers about the market, stating there are approximately 185,000 US construction firms and estimating that 60,000 could benefit from the software based on recurring invoices.
  3. Traction: The pitch cites specific traction by mentioning two regional contractors that are currently paying for a pilot program, demonstrating real engagement with the product.

====== PITCH F: classified as Deep Tech ======
Reasoning: The startup AeroWeave is focused on developing a novel manufacturing process for lightweight structural panels, which is a specific piece of technology designed to enhance electric delivery vans. Their competitive advantage comes from a proprietary metho

**Final draft:**

We are AeroWeave, and we are creating lightweight structural panels specifically designed for electric delivery vans. Our unique advantage lies in our manufacturing process, which carefully places reinforcing fibers exactly where strength is needed. This innovative approach allows us to significantly reduce the weight of the panels without the need for thicker materials. We have a pending patent application that protects this technology, alongside specialized production knowledge that we believe will take competitors several years to replicate. We are currently in discussions with FleetWorks, a regional electric-van manufacturer, which is evaluating an early version of our panel for its next prototype vehicle and has agreed to pay for a materials test.


Final verdict: The pitch successfully meets all criteria, demonstrating a strong technical edge, defensibility, and a clear first customer.

Per-criterion feedback from the last evaluation:
  1. Technical edge: The pitch clearly explains AeroWeave's unique manufacturing process that allows for lightweight structural panels, making it accessible for non-specialists to understand how they differentiate from competitors.
  2. Defensibility: The pitch discusses the pending patent application and specialized production knowledge, providing clear details on what protects the company and suggests that it will take competitors years to replicate this technology.
  3. First customer: The pitch identifies FleetWorks as a specific first buyer who is evaluating the product and has agreed to pay for a materials test, fulfilling this criterion well.


# Analysis

| Pitch | # of iterations |
| - | - |
| A | 1 |
| B | 3 |
| C | 3 |
| D | 2 |
| E | 1 |
| F | 1 |


**Conclude:** PITCH B and PITCH C have the most iterations
* PITCH B: did not pass all of the criteria
    > Customer pain: The pitch provides a real workflow that small business owners dislike and illustrates the frustrations with scheduling, but lacks a specific named scenario that showcases this issue.

    > Market sizing: The pitch estimates the number of potential buyers based on industry reports and government data, which is good, but does not provide an exact number or detailed methodology for the estimation.

    > Traction: The pitch mentions a pilot agreement but fails to name a specific buyer, hence lacking a specific signal of traction.
* PITCH C: did not pass market sizing and traction
    > Market sizing: The pitch provides an estimate of 100,000 potential buyers along with a source, but it lacks clarity on how this figure was derived from the broader statistic of 1 million dining establishments.

    > Traction: The pitch mentions securing a first customer but fails to provide a specific name or any concrete traction signal such as a signed letter of intent, which diminishes its credibility.

    Pitch C final draft ends with `[FIRST CUSTOMER NEEDED: founder must name a specific first buyer]`. The evaluator's final per-criterion feedback explains why tThe pitch fails on this criterion, as it does not name a specific first buyer but instead includes a placeholder for this information, making it unclear who the first customer would be
    
    This shows the real limit as the loop can only rewrite what's already true. The original KitchenSense draft never mentioned a specific first buyer at all, it just claimed the addressable market was "massive" and that the company was "uniquely positioned to capture significant share," which is exactly the kind of vague language the rubric meant
    
    More iterations would not help, the fix isn't better phrasing, it's the founder going out and getting a real early customer, then bringing that fact back into the draft

# Pitchwise Partner Memo

## Agentic Coaching System

The prototype is designed to handle the first-pass review of founder pitches before they reach the partners. Each draft moves through a coaching and evaluation loop: the coach revises the pitch, and an evaluator checks the revised draft against the Pitchwise playbook before deciding whether it should pass or go through another round of revision.

This structure is intended to bring pitches to a consistent baseline quality while reserving partner time for the higher-value judgment, strategy, and founder coaching that the system cannot replace.

## Rubric-Grounded Scoring

The evaluator does not rely on the model's general understanding of what makes a strong startup pitch. Instead, it retrieves the relevant Pitchwise rubric from a centrally managed file and uses that rubric as the sole standard for evaluation.

Because the rubric is stored separately from the model prompts, the partners can update the playbook without changing the underlying system. If a criterion is revised or a new requirement is added, future pitch evaluations automatically use the updated standard. This reduces the risk of outdated versions of the Pitchwise playbook being embedded across different prompts or components of the system.

The system also matches each pitch to the appropriate company category: consumer product, B2B SaaS, or deep tech, so that the evaluation reflects the standards Pitchwise applies to that type of company.

## Pass/Fail Evaluation

A pitch passes only when all criteria in the matched rubric are clearly satisfied. A weakness in any required criterion sends the draft back through the coaching loop rather than allowing stronger areas to compensate for it.

The evaluator also provides a specific explanation for each score instead of returning only a pass/fail decision. This gives partners visibility into why a draft passed or where it continues to fall short, making the system's reasoning easier to review and audit.

## Safeguards Against Unsupported Claims

The system is designed not to invent facts in order to improve a pitch.

If a founder has not provided information required by the rubric—such as a first customer, traction metric, or market-size estimate—the system identifies that gap rather than generating a plausible-sounding number or claim. Missing information is represented with a clearly marked placeholder so that partners and founders can distinguish between a writing problem and a missing business fact.

This is especially important for a first-pass screening tool. A polished pitch should not appear stronger simply because the AI filled in evidence that the founder never provided.

## Prototype Limitations

Two of the six test pitches did not reach a passing score after three revision cycles. In both cases, the evaluator continued to identify weaknesses related to traction or market sizing.

The coaching loop could improve the structure and wording of those pitches, but it could not resolve the underlying issue because the required evidence was absent from the founder's original information. Additional revision cycles therefore produced clearer placeholders rather than substantively stronger pitches.

This behavior highlights an important boundary of the system. Some pitch weaknesses are communication problems that AI can help address; others reflect missing evidence, customer validation, or business development work that cannot be solved through rewriting.

For Pitchwise, that boundary is useful. Rather than allowing a well-written draft to mask a weak underlying case, the system surfaces the point at which the founder needs to gather additional evidence before further pitch coaching will be valuable. The prototype therefore acts as a first-pass quality filter, while leaving the higher-level judgment and founder development work to the partners.
